---

## ToolRunTime

도구는 에이전트 상태, 런타임 컨텍스트 및 장기 메모리에 액세스할 수 있을 때 가장 강력합니다. 이를 통해 도구는 컨텍스트 인식 결정을 내리고, 응답을 개인화하며, 대화 전반에 걸쳐 정보를 유지할 수 있습니다.

`ToolRuntime` 매개변수를 통해 다음 런타임 정보에 액세스할 수 있습니다:

| 속성 | 설명 |
|:---|:---|
| **state** | 실행을 통해 흐르는 변경 가능한 데이터 (메시지, 카운터, 커스텀 필드) |
| **context** | 사용자 ID, 세션 세부 정보 등 불변 구성 정보 |
| **store** | 대화 전반에 걸친 영구 장기 메모리 |
| **stream_writer** | 도구 실행 중 커스텀 업데이트 스트리밍 |
| **config** | 실행을 위한 RunnableConfig |
| **tool_call_id** | 현재 도구 호출의 고유 ID |

---
### ToolRuntime 사용

`ToolRuntime`을 사용하면 단일 매개변수로 모든 런타임 정보에 액세스할 수 있습니다. 도구 시그니처에 `runtime: ToolRuntime`을 추가하면 LLM에는 노출되지 않고 자동으로 주입됩니다.

`runtime.state`를 통해 현재 그래프 상태에 접근하고, `runtime.context`를 통해 컨텍스트 정보에 접근할 수 있습니다.

아래 코드는 ToolRuntime을 사용하여 상태와 컨텍스트에 접근하는 도구 예시입니다.

In [1]:
from feature.ToolRuntimeContext import CustomContext, ToolRuntimeAgent
from langchain_core.messages import HumanMessage

workflow = ToolRuntimeAgent()

In [2]:
from langgraph.graph.state import RunnableConfig


config = {"configurable": {"thread_id": "3"}}
inputs = {"messages": [{"role": "user", "content": "내가 좋아하는 음식 알려줘"}]}

_ = workflow.invoke(
    inputs=inputs,
    config=RunnableConfig(config),
    context=CustomContext(user_preferences={"food": "pizza"}),
)


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  get_user_preference (call_r9z5ncMXRsRoRh5kNypoHZqT)
 Call ID: call_r9z5ncMXRsRoRh5kNypoHZqT
  Args:
    preference_name: food

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: get_user_preference

pizza


/Users/jyuka/workspace/LangGraph-Edu/.venv/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=CustomContext(user_preferences={'food': 'pizza'}), input_type=CustomContext])
  return self.__pydantic_serializer__.to_python(



🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

당신이 좋아하는 음식은 피자입니다!


---

### 컨텍스트

`runtime.context`를 통해 사용자 ID, 세션 정보, 애플리케이션별 구성 등 불변 컨텍스트 데이터에 액세스할 수 있습니다. 컨텍스트는 에이전트 호출 시 전달되며, 도구 실행 중에 변경되지 않습니다.

아래 코드는 사용자 컨텍스트를 활용하여 계정 정보를 조회하는 도구 예시입니다.

In [1]:
from feature.ToolRuntimeContext import CustomContext, ToolRuntimeAgent
from langchain_core.messages import HumanMessage

workflow = ToolRuntimeAgent()

In [2]:
_ = workflow.invoke(
    inputs={"messages": [{"role": "user", "content": "What's my current balance?"}]},
    context=CustomContext(user_id="user123"),
)


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  get_account_info (call_4PizOZ0u7W1alQgeg9hAAFfC)
 Call ID: call_4PizOZ0u7W1alQgeg9hAAFfC
  Args:

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: get_account_info

Account holder: Alice Johnson
Type: Premium
Balance: $5000

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

Your current balance is $5000.
